In [3]:
"""
2024 가공식품 소비자태도조사(가구용)
- 코드북 라벨 매핑
- 페르소나 생성
- 제품 카테고리 선호도 산출(순위 가중)
- 페르소나별 / 개인별 Top-N 추천
"""

import re
import pandas as pd
from collections import defaultdict

# ==============================
# 🔧 CONFIG: 파일 경로
# ==============================
RAW_FILE = "[통계원시자료] 2024년 가공식품 소비자태도조사_가구용.xlsx"
CODEBOOK_FILE = "[코드북] 2024년 가공식품 소비자태도조사_가구용.xlsx"

# 코드북의 컬럼명(여기만 코드북 실제 컬럼명으로 맞추세요)
VAR_COL   = "변수명"        # 예: 변수명
CODE_COL  = "값(Value)"     # 예: 값(Value)
LABEL_COL = "라벨(Label)"   # 예: 라벨(Label)
CODEBOOK_SHEET = 0          # 시트명 또는 인덱스

# ==============================
# 🔧 CONFIG: 페르소나 구성 변수
#   (코드북에서 실제 변수명 확인해 아래를 수정)
# ==============================
PERSONA_CONFIG = {
    "age": "HQ2" or "SQ2",      # 가구주 또는 응답자 출생년도/연령대 중 하나
    "gender": "SQ3",            # 응답자 성별
    "family": "DQ1",            # 가족 구성 형태
    "income": "DQ4",            # 가구 월평균 소득 구간
    "lifestyle_1": "F1",        # 라이프스타일(총괄 문항, 없으면 F1~ 묶음 중 대표)
    # 필요 시 더 추가
}

# ==============================
# 🔧 CONFIG: 선호도/구매 카테고리 추출 변수(순위형)
#   - 가중치: 1순위=3, 2순위=2, 3순위=1 (필요시 조정)
#   - 코드북에서 실제 변수명 접미를 확인해서 아래 리스트 수정
# ==============================
RANK_WEIGHT = {1:3, 2:2, 3:1}

# 지출 비중이 큰 품목군(예: A4 1~3순위)
SPEND_TOP_VARS = ["A4_1", "A4_2", "A4_3"]

# 온라인에서 많이 구입하는 품목군(예: A12 1~3순위)
ONLINE_TOP_VARS = ["A12_1", "A12_2", "A12_3"]

# 간편식(HMR) 주로 구입하는 품목(예: C8 1~3순위)
HMR_TOP_VARS = ["C8_1", "C8_2", "C8_3"]

# 건강기능식품 많이 구입(예: D5 1~2순위)
HFS_TOP_VARS = ["D5_1", "D5_2"]

# (선택) ‘가공식품 전반’ 선호특성 문항이 순위형이면 여기에 추가
# PREF_TOP_VARS = ["F7_xxx_1", "F7_xxx_2", "F7_xxx_3"]


# ==============================
# 1) 데이터 / 코드북 로드 & 라벨 매핑
# ==============================
df_raw = pd.read_excel(RAW_FILE)
df_code = pd.read_excel(CODEBOOK_FILE, sheet_name=CODEBOOK_SHEET)

# 매핑 딕셔너리 생성
mapping_dict = {}
for var in df_code[VAR_COL].dropna().unique():
    sub = df_code[df_code[VAR_COL] == var]
    code_map = dict(zip(sub[CODE_COL], sub[LABEL_COL]))
    mapping_dict[var] = code_map

# 라벨 적용
df = df_raw.copy()
for col in df.columns:
    if col in mapping_dict:
        df[col] = df[col].map(mapping_dict[col]).fillna(df[col])

# 응답자 ID 부여(없으면 만들기)
if "RESP_ID" not in df.columns:
    df.insert(0, "RESP_ID", range(1, len(df)+1))

# ==============================
# 2) 페르소나 생성
# ==============================
def safe_col(df, col):
    return df[col] if col in df.columns else pd.Series(["미응답"]*len(df))

persona_parts = []
for k, col in PERSONA_CONFIG.items():
    if col in df.columns:
        persona_parts.append(df[col].astype(str).fillna("미응답").rename(k))
    else:
        persona_parts.append(pd.Series(["미응답"]*len(df), name=k))

df_persona = pd.concat(persona_parts, axis=1)
df_persona.insert(0, "RESP_ID", df["RESP_ID"])

# 간단한 페르소나 태그 결합(필요 시 축약/정규화)
df_persona["persona_tag"] = df_persona.apply(
    lambda r: " / ".join([str(r[k]) for k in df_persona.columns if k not in ["RESP_ID"]]),
    axis=1
)

# ==============================
# 3) 순위형 변수들 → 카테고리 선호 점수화
#    (동일 카테고리가 여러 블록에서 반복 등장해도 점수 누적)
# ==============================
def melt_rank_vars(df, var_list, block_weight=1.0):
    """
    var_list = ["A4_1","A4_2","A4_3"] 같은 1~3순위 컬럼명
    block_weight = 블록 가중치(해당 블록 중요도 부여하고 싶을 때)
    반환: RESP_ID, category, score
    """
    rows = []
    for col in var_list:
        if col not in df.columns:
            continue
        # 순위 추출: 접미 숫자만 가져오기 (예: A4_2 → 2)
        m = re.search(r'_(\d+)$', col)
        rank = int(m.group(1)) if m else None
        if rank is None:  # 접미 없으면 1순위로 가정
            rank = 1
        weight = RANK_WEIGHT.get(rank, 0) * block_weight
        for rid, cat in zip(df["RESP_ID"], df[col].fillna("미응답")):
            if pd.isna(cat) or str(cat).strip() in ["", "미응답"]:
                continue
            rows.append((rid, str(cat), weight))
    return pd.DataFrame(rows, columns=["RESP_ID","category","score"])

# 블록별 데이터 만들고 가중치(중요도) 줄 수 있음(원하면 변경)
pref_frames = []
pref_frames.append(melt_rank_vars(df, SPEND_TOP_VARS, block_weight=1.0))
pref_frames.append(melt_rank_vars(df, ONLINE_TOP_VARS, block_weight=0.8))
pref_frames.append(melt_rank_vars(df, HMR_TOP_VARS,   block_weight=0.9))
pref_frames.append(melt_rank_vars(df, HFS_TOP_VARS,   block_weight=1.1))

# 합치고 개인별 카테고리 점수 합계
df_pref = pd.concat(pref_frames, ignore_index=True)
pref_user = (
    df_pref.groupby(["RESP_ID","category"], as_index=False)["score"]
           .sum()
           .sort_values(["RESP_ID","score"], ascending=[True, False])
)

# ==============================
# 4) 개인별 Top-N 추천
# ==============================
TOPN = 5
def topn_per_user(df_user_pref, n=TOPN):
    # RESP_ID 단위 상위 n
    return df_user_pref.groupby("RESP_ID").head(n).reset_index(drop=True)

user_topn = topn_per_user(pref_user, TOPN)

# 읽기 좋게 pivot (선택)
user_topn_pivot = (
    user_topn.assign(rank=user_topn.groupby("RESP_ID").cumcount()+1)
             .pivot(index="RESP_ID", columns="rank", values="category")
             .rename(columns=lambda c: f"TOP{c}")
             .reset_index()
)

# ==============================
# 5) 페르소나별 선호도 & 추천
# ==============================
# 개인 선호와 페르소나 연결
pref_user_persona = pref_user.merge(df_persona[["RESP_ID","persona_tag"]], on="RESP_ID", how="left")

# 페르소나별 카테고리 점수 합계 → Top-N
persona_pref = (
    pref_user_persona.groupby(["persona_tag","category"], as_index=False)["score"]
                     .sum()
                     .sort_values(["persona_tag","score"], ascending=[True, False])
)

persona_topn = persona_pref.groupby("persona_tag").head(TOPN).reset_index(drop=True)

# (선택) 페르소나별 상위 카테고리 표 형태
persona_topn_pivot = (
    persona_topn.assign(rank=persona_topn.groupby("persona_tag").cumcount()+1)
                .pivot(index="persona_tag", columns="rank", values="category")
                .rename(columns=lambda c: f"TOP{c}")
                .reset_index()
)

# 페르소나 비중도 함께 보고 싶다면:
persona_dist = (
    df_persona["persona_tag"].value_counts()
    .rename_axis("persona_tag")
    .reset_index(name="count")
)
persona_dist["ratio(%)"] = (persona_dist["count"]/len(df_persona)*100).round(2)


# ==============================
# 6) 결과 저장
# ==============================
df.to_excel("01_라벨적용_전체데이터.xlsx", index=False)
df_persona.to_excel("02_페르소나_개별속성.xlsx", index=False)
pref_user.to_excel("03_개인별_카테고리점수.xlsx", index=False)
user_topn.to_excel("04_개인별_TOP{}_카테고리_long.xlsx".format(TOPN), index=False)
user_topn_pivot.to_excel("05_개인별_TOP{}_카테고리_wide.xlsx".format(TOPN), index=False)
persona_pref.to_excel("06_페르소나별_카테고리점수.xlsx", index=False)
persona_topn.to_excel("07_페르소나별_TOP{}_카테고리_long.xlsx".format(TOPN), index=False)
persona_topn_pivot.to_excel("08_페르소나별_TOP{}_카테고리_wide.xlsx".format(TOPN), index=False)
persona_dist.to_excel("09_페르소나_비중.xlsx", index=False)

print("✅ 완료!")
print(" - 01_라벨적용_전체데이터.xlsx")
print(" - 02_페르소나_개별속성.xlsx")
print(" - 03_개인별_카테고리점수.xlsx")
print(" - 04_개인별_TOP{}_카테고리_long.xlsx".format(TOPN))
print(" - 05_개인별_TOP{}_카테고리_wide.xlsx".format(TOPN))
print(" - 06_페르소나별_카테고리점수.xlsx")
print(" - 07_페르소나별_TOP{}_카테고리_long.xlsx".format(TOPN))
print(" - 08_페르소나별_TOP{}_카테고리_wide.xlsx".format(TOPN))
print(" - 09_페르소나_비중.xlsx")


KeyError: '값(Value)'